In [1]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime

# 取得今天的日期，格式化成 YYYYMMDD
today_date = datetime.today().strftime('%Y%m%d')

print(today_date)  # 例如: 20250317


20250414


In [2]:
today_date = int(today_date)


In [3]:
today_date

20250414

In [4]:
# aid = 1 : MLB
input_date = '20250210'
user_agent = ''
headers = {'User-Agent': user_agent}
url = f'https://www.playsport.cc/livescore.php?aid=1&gamedate={input_date}&mode=11'
response = requests.get(url, headers=headers)

# 確認是否成功
if response.status_code == 200:
    print('連線成功')
else:
    print('連線失敗')


連線成功


In [5]:
html_source = response.text
soup = BeautifulSoup(html_source, 'html.parser')

In [6]:
# aid = 1 : MLB
input_date = '20250314'
user_agent = ''
headers = {'User-Agent': user_agent}
url = f'https://www.playsport.cc/livescore.php?aid=1&gamedate={input_date}&mode=11'
response = requests.get(url, headers=headers)

# 確認是否成功
if response.status_code == 200:
    print('連線成功')
else:
    print('連線失敗')


html_source = response.text
soup = BeautifulSoup(html_source, 'html.parser')


# 找到比賽容器
livescore_container = soup.find('div', id='livescoreContainer')

# 確認是否有比賽數據
if not livescore_container or '暫無資料' in livescore_container.text:
    print("沒有比賽數據")
    games = []
else:
    games = []
    try:
        # 找到所有比賽區塊
        for game in soup.find_all('div', class_='js-gameOnbox'):
            team_a = game.get('data-namea', '未知隊伍').strip()  # 客隊
            team_h = game.get('data-nameh', '未知隊伍').strip()  # 主隊

            score_a = game.find('td', class_='big_score', id=lambda x: x and x.endswith('_as_b'))
            score_h = game.find('td', class_='big_score', id=lambda x: x and x.endswith('_hs_b'))
            score_a = score_a.text.strip() if score_a else '0'
            score_h = score_h.text.strip() if score_h else '0'

            # 每局得分
            innings_a = []
            innings_h = []
            for i in range(1, 10):  # 棒球最多9局
                inning_a = game.find('td', id=lambda x: x and x.endswith(f'_as{i}'))
                inning_h = game.find('td', id=lambda x: x and x.endswith(f'_hs{i}'))
                innings_a.append(inning_a.text.strip() if inning_a else '0')
                innings_h.append(inning_h.text.strip() if inning_h else '0')

            # R, H, E (總得分、安打數、失誤數)
            total_r_a = game.find('td', id=lambda x: x and x.endswith('_asr'))
            total_h_a = game.find('td', id=lambda x: x and x.endswith('_ash'))
            total_e_a = game.find('td', id=lambda x: x and x.endswith('_ase'))

            total_r_h = game.find('td', id=lambda x: x and x.endswith('_hsr'))
            total_h_h = game.find('td', id=lambda x: x and x.endswith('_hsh'))
            total_e_h = game.find('td', id=lambda x: x and x.endswith('_hse'))

            game_data = {
                'team_a': team_a,
                'team_h': team_h,
                'score_a': total_r_a.text.strip() if total_r_a else '0',
                'score_h': total_r_h.text.strip() if total_r_h else '0',
                'innings': {
                    'team_a': innings_a,
                    'team_h': innings_h
                },
                'total': {
                    'team_a': {
                        'R': total_r_a.text.strip() if total_r_a else '0',
                        'H': total_h_a.text.strip() if total_h_a else '0',
                        'E': total_e_a.text.strip() if total_e_a else '0'
                    },
                    'team_h': {
                        'R': total_r_h.text.strip() if total_r_h else '0',
                        'H': total_h_h.text.strip() if total_h_h else '0',
                        'E': total_e_h.text.strip() if total_e_h else '0'
                    }
                }
            }

            games.append(game_data)

    except Exception as e:
        print(f"發生錯誤: {e}")
        games = []

連線成功


In [7]:
games

[{'team_a': '紅雀',
  'team_h': '太空人',
  'score_a': '1',
  'score_h': '1',
  'innings': {'team_a': ['0', '0', '1', '0', '0', '0', '0', '0', '0'],
   'team_h': ['0', '0', '1', '0', '0', '0', '0', '0', '0']},
  'total': {'team_a': {'R': '1', 'H': '3', 'E': '0'},
   'team_h': {'R': '1', 'H': '4', 'E': '1'}}},
 {'team_a': '國民',
  'team_h': '光芒',
  'score_a': '4',
  'score_h': '14',
  'innings': {'team_a': ['3', '0', '1', '0', '0', '0', '0', '0', '0'],
   'team_h': ['0', '0', '1', '5', '7', '0', '0', '1', 'X']},
  'total': {'team_a': {'R': '4', 'H': '7', 'E': '0'},
   'team_h': {'R': '14', 'H': '15', 'E': '0'}}},
 {'team_a': '海盜',
  'team_h': '雙城',
  'score_a': '3',
  'score_h': '15',
  'innings': {'team_a': ['0', '0', '0', '0', '0', '0', '0', '0', '3'],
   'team_h': ['0', '0', '0', '0', '5', '9', '1', '0', 'X']},
  'total': {'team_a': {'R': '3', 'H': '6', 'E': '2'},
   'team_h': {'R': '15', 'H': '15', 'E': '1'}}},
 {'team_a': '勇士',
  'team_h': '費城人',
  'score_a': '9',
  'score_h': '16',
  'i